<a href="https://colab.research.google.com/github/solimanmohamed7177-arch/-Weather-Classification-PSO-GWO-Optimization-Project/blob/main/RAG_LangChain_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building and Analyzing a Retrieval-Augmented Generation (RAG) System using LangChain

**A Hands-On University Lab Assignment**

---

**Course topic:** Natural Language Processing / Large Language Models
**Estimated time:** 2–4 hours
**Environment:** Google Colab (free GPU tier) or Kaggle Notebooks (free GPU tier)
**Cost:** $0 — every model and tool used here is open-source and runs locally on the notebook's own compute.

---

## Table of Contents

1. [Introduction](#1)
2. [Installation](#2)
3. [Project Configuration](#3)
4. [Dataset](#4)
5. [Document Loaders](#5)
6. [Chunking](#6)
7. [Embedding Models](#7)
8. [Vector Database](#8)
9. [Retriever](#9)
10. [Prompt Engineering](#10)
11. [LLM](#11)
12. [Building the Complete RAG Pipeline](#12)
13. [Experiments](#13)
14. [Optional Evaluation](#14)
15. [Student Experiment Table](#15)
16. [Reflection Questions](#16)
17. [Bonus Challenges](#17)

---

> **How to use this notebook:** Run the cells **in order, from top to bottom**. Every section builds on
> the previous one through shared variables defined in Section 3 (`CONFIG`). To re-run an experiment
> with different settings, change a value in `CONFIG`, re-run that cell, and re-run every cell below it.
> You do **not** need to edit code outside the clearly marked "change me" locations.


<a id="1"></a>
# 1. Introduction

## 1.1 What is Retrieval-Augmented Generation (RAG)?

A Large Language Model (LLM) generates text based only on the knowledge it memorized during training.
**Retrieval-Augmented Generation (RAG)** is a technique that gives an LLM access to an *external*,
up-to-date, and verifiable knowledge base at answer time. Instead of asking the LLM to answer purely
from memory, we first **retrieve** relevant text passages from a document collection, and then **feed
those passages into the LLM's prompt** as context before asking it to answer.

RAG turns a "closed-book exam" (the LLM answering from memory alone) into an "open-book exam" (the LLM
answering while looking at relevant pages you handed it).

## 1.2 Why do LLMs hallucinate?

LLMs are trained to predict the statistically most likely next token, not to consult a ground-truth
database. This causes **hallucination** — confident-sounding but factually wrong output — for several
reasons:

- **Parametric memory only:** the model can only "know" what was present (and well-represented) in its
  training data.
- **Knowledge cutoff:** anything that happened after training is unknown to the model.
- **No source verification:** the model cannot check whether what it is about to say is actually true;
  it only knows what is *probable* given the text so far.
- **Long-tail / niche knowledge:** rare facts (e.g., a specific company's internal policy, a specific
  textbook's Chapter 7) are rarely well represented in general training corpora.

## 1.3 Why does RAG reduce hallucination?

By grounding generation in retrieved, verifiable text, RAG:

- Gives the model **concrete evidence** to summarize/answer from, rather than requiring it to invent facts.
- Allows the model to say "I don't know" or "the provided context does not mention this" — this is only
  possible when there *is* a context to compare against.
- Makes it possible to **cite sources**, since we know exactly which document chunks were used.
- Keeps the knowledge **up to date** without retraining the model — you just update the document
  collection.

## 1.4 Fine-Tuning vs. RAG

| Aspect | Fine-Tuning | RAG |
|---|---|---|
| Updates knowledge | Bakes facts into model weights | Injects facts into the prompt at query time |
| Cost to update knowledge | Expensive — requires retraining | Cheap — just re-index new documents |
| Source attribution | Not possible | Possible (chunks + metadata) |
| Risk of hallucination | Still present | Reduced, not eliminated |
| Good for | Teaching a model a new *style*, *format*, or *skill* | Teaching a model new *facts* |
| Latency | No extra latency at inference | Extra latency (retrieval step) |

In practice, the two are complementary rather than competing: you can fine-tune a model to be better at
*using* retrieved context, while still relying on RAG to supply the facts.

## 1.5 Advantages of RAG

- Reduces hallucination by grounding answers in real text.
- Knowledge base can be updated instantly (no retraining).
- Enables source citation and traceability.
- Works with small, cheap, even open-source LLMs, since the LLM's job is reduced to "read and
  summarize/answer", not "recall from memory".
- Domain adaptation without touching model weights.

## 1.6 Limitations of RAG

- Answer quality is bounded by **retrieval quality** — if the retriever fetches the wrong chunks, the
  LLM will confidently answer from the wrong context ("garbage in, garbage out").
- Struggles with questions that require reasoning **across many documents** or the **entire corpus**
  (e.g., "how many documents mention X?").
- Chunking can **split context awkwardly**, cutting a fact in half across two chunks.
- Adds system complexity: a vector database, an embedding model, and a retrieval step, all of which can
  fail independently.
- Still does not guarantee factual correctness — the LLM can still misread or misrepresent the retrieved
  context.

## 1.7 Real-world applications

- Enterprise knowledge-base chatbots (HR policy Q&A, internal documentation search).
- Customer support assistants grounded in product manuals.
- Legal and medical document question-answering.
- Academic literature review assistants.
- Coding assistants grounded in a specific codebase's documentation.

## 1.8 End-to-end RAG workflow

```
   Documents (PDF, TXT, DOCX, MD, ...)
            │
            ▼
      [ 1. Loader ]            → parses raw files into LangChain Document objects
            │
            ▼
      [ 2. Chunking ]          → splits long documents into small, retrievable pieces
            │
            ▼
      [ 3. Embeddings ]        → converts each chunk into a numeric vector
            │
            ▼
      [ 4. Vector Store ]      → stores vectors for fast similarity search (Chroma)
            │
            ▼
      [ 5. Retriever ]         → given a question, finds the top-k most similar chunks
            │
            ▼
      [ 6. Prompt Template ]   → combines the question + retrieved chunks into one prompt
            │
            ▼
      [ 7. LLM ]               → generates an answer conditioned on the prompt
            │
            ▼
         Answer
```

This notebook builds every one of these seven stages from scratch, one section at a time, so that by the
end you understand — and can modify — every moving part of a RAG system.

## 1.9 Learning objectives

By the end of this notebook, you will be able to:

1. Explain what RAG is and why it helps reduce hallucination.
2. Load documents of multiple formats (PDF, TXT, DOCX, Markdown) using LangChain loaders.
3. Explain and implement text chunking strategies, and reason about chunk size/overlap trade-offs.
4. Generate and compare sentence embeddings using open-source embedding models.
5. Build, persist, and query a vector database (Chroma).
6. Implement a retriever and reason about top-k and precision/recall trade-offs.
7. Design an effective RAG prompt template.
8. Load and run an open-source instruction-tuned LLM locally with HuggingFace `transformers`.
9. Assemble all components into a complete, runnable RAG pipeline using modern LangChain (LCEL) syntax.
10. Run controlled experiments varying chunking, embeddings, retrieval, and LLM parameters, and critically
    analyze the results.


<a id="2"></a>
# 2. Installation

## What we are installing and why

| Package | Why we need it |
|---|---|
| `langchain` | Core orchestration framework: prompts, chains, retrievers. |
| `langchain-community` | Community-maintained integrations (loaders, vector stores glue). |
| `langchain-huggingface` | Official LangChain ↔ HuggingFace integration (embeddings + LLM wrappers). |
| `langchain-chroma` | LangChain's Chroma vector-store integration. |
| `chromadb` | The actual open-source vector database engine. |
| `sentence-transformers` | Backend used to run the embedding models. |
| `transformers` | Loads and runs the open-source LLM. |
| `accelerate` | Efficient model loading/placement on GPU or CPU. |
| `pypdf` | Parses PDF files for `PyPDFLoader`. |
| `docx2txt` | Parses `.docx` files for `Docx2txtLoader`. |
| `beautifulsoup4` | HTML parsing (used by some loaders / bonus HTML loader). |
| `unstructured` | Powers `UnstructuredMarkdownLoader` and other flexible format loaders. |
| `python-docx` | Used only to *generate* our demo `.docx` sample file. |
| `fpdf2` | Used only to *generate* our demo `.pdf` sample file. |
| `bitsandbytes` | Optional 4-bit/8-bit quantization to reduce LLM memory usage. |

> **Tip:** On Colab/Kaggle this cell can take a few minutes the first time. You only need to run it once
> per session (unless the runtime restarts).

> **Common mistake:** Forgetting to restart the runtime after installing new packages can cause
> `ImportError`s. If you hit an import error right after installing, use *Runtime → Restart Runtime*
> (Colab) and re-run from the top.


In [1]:
# Install all required open-source packages. Everything here is free and open-source —
# no API keys, no paid services.
%pip install -q -U \
    langchain \
    langchain-community \
    langchain-huggingface \
    langchain-chroma \
    chromadb \
    sentence-transformers \
    transformers \
    accelerate \
    pypdf \
    docx2txt \
    beautifulsoup4 \
    "unstructured[md]" \
    python-docx \
    fpdf2 \
    bitsandbytes \
    rank_bm25

print("Installation complete.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 26.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 17.5 MB/s eta 0:00:00


In [2]:
# Verify that the key libraries imported successfully and print their versions.
# This is good practice: it makes bug reports reproducible and confirms our environment is sane.

import importlib

packages_to_check = [
    "langchain",
    "langchain_community",
    "langchain_huggingface",
    "langchain_chroma",
    "chromadb",
    "sentence_transformers",
    "transformers",
    "accelerate",
    "torch",
]

print(f"{'Package':<25}{'Version'}")
print("-" * 40)
for pkg_name in packages_to_check:
    try:
        module = importlib.import_module(pkg_name)
        version = getattr(module, "__version__", "unknown")
        print(f"{pkg_name:<25}{version}")
    except ImportError as e:
        print(f"{pkg_name:<25}NOT INSTALLED ({e})")

# Check for GPU availability — RAG will run on CPU too, just more slowly.
import torch
device_available = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nDetected compute device: {device_available}")
if device_available == "cpu":
    print("No GPU detected. The notebook will still run, but LLM generation will be slower.")
    print("On Colab: Runtime > Change runtime type > T4 GPU (free tier) is recommended.")

Package                  Version
----------------------------------------
langchain                1.3.14
langchain_community      0.4.2
langchain_huggingface    unknown
langchain_chroma         unknown
chromadb                 1.5.9
sentence_transformers    5.6.1
transformers             5.14.1
accelerate               1.14.0
torch                    2.11.0+cu128

Detected compute device: cuda


<a id="3"></a>
# 3. Project Configuration

This section centralizes **every hyperparameter** used in the notebook into a single `CONFIG` dictionary.

> **This is the only place you should need to edit to run new experiments.** Change a value here,
> re-run this cell, then re-run the cells in the sections that depend on it (they are clearly explained
> in each section).

| Variable | Meaning |
|---|---|
| `DATA_PATH` | Folder containing the source documents to index. |
| `LLM_MODEL` | HuggingFace model id of the instruction-tuned LLM to use for generation. |
| `EMBEDDING_MODEL` | HuggingFace model id of the sentence-embedding model. |
| `CHUNK_SIZE` | Maximum number of characters per text chunk. |
| `CHUNK_OVERLAP` | Number of overlapping characters between consecutive chunks. |
| `TOP_K` | Number of chunks retrieved per query. |
| `DEVICE` | `"cuda"` if a GPU is available, else `"cpu"`. |
| `PERSIST_DIRECTORY` | Folder where the Chroma vector database is persisted to disk. |
| `RANDOM_SEED` | Seed for reproducibility. |


In [3]:
import os
import random
import numpy as np
import torch

CONFIG = {
    # ---- Data ----
    "DATA_PATH": "./data",                                   # folder holding source documents
    "PERSIST_DIRECTORY": "./chroma_db",                       # where the vector DB is saved on disk

    # ---- Models (change ONLY these two lines to swap models) ----
    "LLM_MODEL": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",         # <-- change me to switch LLM
    "EMBEDDING_MODEL": "sentence-transformers/all-MiniLM-L6-v2",  # <-- change me to switch embeddings

    # ---- Chunking ----
    "CHUNK_SIZE": 500,       # try: 200 / 500 / 1000
    "CHUNK_OVERLAP": 50,     # try: 0 / 50 / 100

    # ---- Retrieval ----
    "TOP_K": 4,               # try: 2 / 4 / 6 / 8

    # ---- Generation ----
    "TEMPERATURE": 0.3,
    "MAX_NEW_TOKENS": 256,
    "DO_SAMPLE": True,
    "TOP_P": 0.9,
    "REPETITION_PENALTY": 1.1,

    # ---- Misc ----
    "RANDOM_SEED": 42,
}

# Automatically detect whether a GPU is available.
CONFIG["DEVICE"] = "cuda" if torch.cuda.is_available() else "cpu"

# Set random seeds everywhere for reproducibility.
random.seed(CONFIG["RANDOM_SEED"])
np.random.seed(CONFIG["RANDOM_SEED"])
torch.manual_seed(CONFIG["RANDOM_SEED"])

# Make sure our working folders exist.
os.makedirs(CONFIG["DATA_PATH"], exist_ok=True)
os.makedirs(CONFIG["PERSIST_DIRECTORY"], exist_ok=True)

print("Current configuration:")
for key, value in CONFIG.items():
    print(f"  {key:<20} = {value}")

Current configuration:
  DATA_PATH            = ./data
  PERSIST_DIRECTORY    = ./chroma_db
  LLM_MODEL            = TinyLlama/TinyLlama-1.1B-Chat-v1.0
  EMBEDDING_MODEL      = sentence-transformers/all-MiniLM-L6-v2
  CHUNK_SIZE           = 500
  CHUNK_OVERLAP        = 50
  TOP_K                = 4
  TEMPERATURE          = 0.3
  MAX_NEW_TOKENS       = 256
  DO_SAMPLE            = True
  TOP_P                = 0.9
  REPETITION_PENALTY   = 1.1
  RANDOM_SEED          = 42
  DEVICE               = cuda


<a id="4"></a>
# 4. Dataset

## 4.1 Supported formats

This notebook supports four common document formats out of the box:

| Format | Advantages | Disadvantages |
|---|---|---|
| **PDF** (`.pdf`) | Ubiquitous for reports, papers, books; preserves layout. | Text extraction can be messy (columns, tables, headers/footers); scanned PDFs need OCR. |
| **TXT** (`.txt`) | Simplest possible format; trivial to parse; no extraction errors. | No structure (headings, metadata) to exploit. |
| **DOCX** (`.docx`) | Common for reports/assignments; preserves some structure (headings, styles). | Requires a parser library; complex layouts (tables, images) can be lossy. |
| **Markdown** (`.md`) | Lightweight, human-readable, structure via headings is easy to exploit. | Less common as a "real world" source format outside of docs/wikis/READMEs. |

## 4.2 Our demo corpus

For this assignment we auto-generate a **small, self-contained demo corpus** about RAG itself, so the
notebook is fully reproducible without external downloads. The corpus intentionally contains structured
sections (headings, lists) so you can see how chunking interacts with document structure.

> **To use your own files:** simply upload your PDFs/TXT/DOCX/MD files into the `CONFIG["DATA_PATH"]`
> folder (`./data` by default) — e.g. via the Colab file browser — and skip the generation cell below.
> Everything downstream (loaders, chunking, embeddings, retrieval) will work unchanged.


In [4]:
# Generate a small, self-contained demo dataset covering PDF, TXT, DOCX, and Markdown formats.
# All four files describe different (but related) aspects of RAG, so retrieval has a real chance
# to fetch DIFFERENT chunks for different questions.

from docx import Document as DocxDocument
from fpdf import FPDF

data_path = CONFIG["DATA_PATH"]

# ---------- 1. TXT file ----------
txt_content = '''Chapter 1: Introduction to Retrieval-Augmented Generation

Retrieval-Augmented Generation (RAG) is a technique that combines a retrieval system with a generative
language model. Instead of relying solely on knowledge memorized during training, a RAG system searches
an external knowledge base at query time and supplies the most relevant passages to the language model
as additional context.

The main components of a RAG system are: a document loader, a text splitter, an embedding model, a
vector database, a retriever, a prompt template, and a language model. Each component can be swapped
independently, which makes RAG systems highly modular.

RAG was popularized by the 2020 paper "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks"
by Lewis et al., which showed that combining a retriever with a generator outperformed purely generative
models on open-domain question answering benchmarks.
'''
with open(f"{data_path}/01_introduction.txt", "w", encoding="utf-8") as f:
    f.write(txt_content)

# ---------- 2. Markdown file ----------
md_content = '''# Chapter 2: Advantages of RAG

RAG offers several concrete advantages over relying on a language model's parametric memory alone:

- **Up-to-date knowledge**: the knowledge base can be updated at any time without retraining the model.
- **Source attribution**: because retrieved chunks carry metadata (source file, page number), answers
  can be traced back to their origin.
- **Reduced hallucination**: grounding generation in retrieved text reduces (but does not eliminate)
  fabricated answers.
- **Lower cost than fine-tuning**: updating a vector database is far cheaper than retraining or
  fine-tuning a large model.

## Limitations

RAG is not a silver bullet. If the retriever fetches irrelevant chunks, the language model may still
produce an incorrect answer, because it is only as good as the context it is given. Chunking strategy,
embedding model quality, and retriever configuration all directly affect the quality of the final answer.
'''
with open(f"{data_path}/02_advantages.md", "w", encoding="utf-8") as f:
    f.write(md_content)

# ---------- 3. DOCX file ----------
docx_doc = DocxDocument()
docx_doc.add_heading("Chapter 3: Chunking Strategies", level=1)
docx_doc.add_paragraph(
    "Chunking is the process of splitting long documents into smaller pieces before embedding them. "
    "This is necessary because embedding models and language models have a limited context window, "
    "and because smaller, focused chunks tend to produce more precise retrieval results than very "
    "long ones."
)
docx_doc.add_heading("Chunk size", level=2)
docx_doc.add_paragraph(
    "Chunk size controls how many characters (or tokens) go into each chunk. Small chunks are precise "
    "but may lack context; large chunks preserve context but can dilute the embedding with irrelevant "
    "information and reduce retrieval precision."
)
docx_doc.add_heading("Chunk overlap", level=2)
docx_doc.add_paragraph(
    "Chunk overlap controls how many characters are shared between two consecutive chunks. Overlap "
    "helps avoid cutting a sentence or idea awkwardly in half at a chunk boundary, at the cost of "
    "some redundancy in the vector database."
)
docx_doc.save(f"{data_path}/03_chunking.docx")

# ---------- 4. PDF file ----------
pdf = FPDF()
pdf.add_page()
pdf.set_font("Helvetica", size=14)
pdf.multi_cell(0, 10, "Chapter 4: Embeddings and Vector Search")
pdf.set_font("Helvetica", size=11)
pdf.ln(4)
pdf.multi_cell(
    0, 8,
    "An embedding model converts a piece of text into a fixed-length numeric vector such that "
    "semantically similar texts are mapped to nearby vectors in that vector space. Semantic similarity "
    "is typically measured using cosine similarity between two vectors.\n\n"
    "A vector database stores these embeddings and provides an efficient way to search for the vectors "
    "closest to a given query vector, which is known as approximate nearest neighbor (ANN) search. "
    "Chroma is a popular open-source vector database that is easy to run locally and integrates "
    "directly with LangChain.\n\n"
    "The author of this course material is the NLP Teaching Team.",
)
pdf.output(f"{data_path}/04_embeddings.pdf")

print("Demo dataset created in:", data_path)
for fname in sorted(os.listdir(data_path)):
    fpath = os.path.join(data_path, fname)
    print(f"  {fname:<25}({os.path.getsize(fpath)} bytes)")

Demo dataset created in: ./data
  01_introduction.txt      (912 bytes)
  02_advantages.md         (951 bytes)
  03_chunking.docx         (37029 bytes)
  04_embeddings.pdf        (1447 bytes)


<a id="5"></a>
# 5. Document Loaders

## 5.1 What loaders do

A LangChain **document loader** reads a raw file and converts it into one or more `Document` objects,
each with:

- `page_content`: the extracted plain text.
- `metadata`: a dictionary carrying information like the source filename, page number, etc.

## 5.2 Loaders used in this notebook

| Loader | File type(s) | Strengths | Weaknesses |
|---|---|---|---|
| `PyPDFLoader` | `.pdf` | Fast, one `Document` per page, keeps page numbers in metadata. | Struggles with complex layouts (multi-column, tables), no OCR for scanned PDFs. |
| `TextLoader` | `.txt` | Extremely simple and reliable — plain text in, plain text out. | No structure awareness at all. |
| `Docx2txtLoader` | `.docx` | Simple, lightweight, good for text-heavy Word documents. | Loses formatting/structure (headings become plain text); no tables/images. |
| `UnstructuredMarkdownLoader` | `.md` | Understands Markdown structure (headings, lists) to some degree. | Slightly heavier dependency (`unstructured`); behavior can vary by version. |
| `DirectoryLoader` | any (delegates to the loaders above) | Loads an entire folder in one call, mixing multiple formats via a glob pattern + loader mapping. | Errors in one file can stop the whole batch unless `silent_errors=True`. |

We load each format individually first (so you can see each loader's specific behavior), and then show
how `DirectoryLoader` can load the whole folder at once.


In [5]:
from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    Docx2txtLoader,
    UnstructuredMarkdownLoader,
    DirectoryLoader,
)

def load_and_report(loader, label):
    '''Run a loader, print a short report, and return the list of Document objects.'''
    docs = loader.load()
    print(f"--- {label} ---")
    print(f"  Documents loaded : {len(docs)}")
    for i, doc in enumerate(docs):
        preview = doc.page_content.strip().replace(chr(10), " ")[:120]
        print(f"  [{i}] metadata={doc.metadata}")
        print(f"      preview: {preview}...")
    print()
    return docs

data_path = CONFIG["DATA_PATH"]

pdf_docs = load_and_report(PyPDFLoader(f"{data_path}/04_embeddings.pdf"), "PyPDFLoader (.pdf)")
txt_docs = load_and_report(TextLoader(f"{data_path}/01_introduction.txt", encoding="utf-8"), "TextLoader (.txt)")
docx_docs = load_and_report(Docx2txtLoader(f"{data_path}/03_chunking.docx"), "Docx2txtLoader (.docx)")
md_docs = load_and_report(UnstructuredMarkdownLoader(f"{data_path}/02_advantages.md"), "UnstructuredMarkdownLoader (.md)")

--- PyPDFLoader (.pdf) ---
  Documents loaded : 1
  [0] metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-07-31T20:31:24+00:00', 'source': './data/04_embeddings.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}
      preview: Chapter 4: Embeddings and Vector Search An embedding model converts a piece of text into a fixed-length numeric vector s...

--- TextLoader (.txt) ---
  Documents loaded : 1
  [0] metadata={'source': './data/01_introduction.txt'}
      preview: Chapter 1: Introduction to Retrieval-Augmented Generation  Retrieval-Augmented Generation (RAG) is a technique that comb...

--- Docx2txtLoader (.docx) ---
  Documents loaded : 1
  [0] metadata={'source': './data/03_chunking.docx'}
      preview: Chapter 3: Chunking Strategies  Chunking is the process of splitting long documents into smaller pieces before embedding...

--- UnstructuredMarkdownLoader (.md) ---
  Documents loaded : 1
  [0] metadata={'source': './data/02_advantages.md'}
      preview

In [6]:
# DirectoryLoader can load an entire folder using a glob pattern + a loader class + loader kwargs.
# Here we demonstrate loading ALL .txt files in the folder in one call (extendable to other formats
# by running DirectoryLoader once per extension and concatenating the results — see the loop below).

loader_config = [
    ("**/*.pdf", PyPDFLoader, {}),
    ("**/*.txt", TextLoader, {"encoding": "utf-8"}),
    ("**/*.docx", Docx2txtLoader, {}),
    ("**/*.md", UnstructuredMarkdownLoader, {}),
]

all_documents = []
for glob_pattern, loader_cls, loader_kwargs in loader_config:
    directory_loader = DirectoryLoader(
        data_path,
        glob=glob_pattern,
        loader_cls=loader_cls,
        loader_kwargs=loader_kwargs,
        show_progress=False,
        silent_errors=True,   # don't let one bad file stop the whole batch
    )
    loaded = directory_loader.load()
    print(f"DirectoryLoader matched '{glob_pattern}': {len(loaded)} document(s)")
    all_documents.extend(loaded)

print(f"\nTotal documents loaded across all formats: {len(all_documents)}")

# --- Loader comparison table ---
print(f"\n{'Format':<10}{'Loader':<28}{'# Documents':<14}{'Notes'}")
print("-" * 80)
print(f"{'PDF':<10}{'PyPDFLoader':<28}{len(pdf_docs):<14}{'1 Document per page'}")
print(f"{'TXT':<10}{'TextLoader':<28}{len(txt_docs):<14}{'1 Document per file'}")
print(f"{'DOCX':<10}{'Docx2txtLoader':<28}{len(docx_docs):<14}{'1 Document per file'}")
print(f"{'MD':<10}{'UnstructuredMarkdownLoader':<28}{len(md_docs):<14}{'1+ Document(s), structure-aware'}")

DirectoryLoader matched '**/*.pdf': 1 document(s)
DirectoryLoader matched '**/*.txt': 1 document(s)
DirectoryLoader matched '**/*.docx': 1 document(s)
DirectoryLoader matched '**/*.md': 1 document(s)

Total documents loaded across all formats: 4

Format    Loader                      # Documents   Notes
--------------------------------------------------------------------------------
PDF       PyPDFLoader                 1             1 Document per page
TXT       TextLoader                  1             1 Document per file
DOCX      Docx2txtLoader              1             1 Document per file
MD        UnstructuredMarkdownLoader  1             1+ Document(s), structure-aware


**Discussion:** Notice that `PyPDFLoader` returns one `Document` *per page*, while the other
loaders return one `Document` *per file*. This matters later: metadata like `page` is only meaningful
for PDF-derived chunks, and it directly affects how many "documents" enter the chunking stage before
any splitting even happens.


<a id="6"></a>
# 6. Chunking

## 6.1 Why chunking exists

Both embedding models and LLMs have a **limited context window** (a maximum number of tokens they can
process at once). Feeding an entire book into either would be impossible, and even if it were possible,
a single embedding vector for an entire book would be too generic to be useful for precise retrieval.
**Chunking** splits documents into smaller, semantically coherent pieces so that:

- Each chunk fits comfortably inside the embedding model's and LLM's context window.
- Each chunk's embedding represents a *focused* topic, improving retrieval precision.
- We can retrieve just the 2–8 chunks that are actually relevant to a question, instead of the whole
  corpus.

## 6.2 Chunk size and chunk overlap

- **Chunk size** — the maximum length (in characters, here) of each chunk. Too small → chunks lack
  enough context to be useful on their own. Too large → chunks become unfocused, hurting retrieval
  precision, and may not fit the context window once several are retrieved together.
- **Chunk overlap** — the number of characters repeated between consecutive chunks. This prevents an
  important sentence from being awkwardly cut in half exactly at a chunk boundary, at the cost of some
  redundant storage.

## 6.3 Splitters used here

- **`RecursiveCharacterTextSplitter`** (recommended default): tries to split on a hierarchy of
  separators (`"\n\n"`, `"\n"`, `" "`, `""`) — it prefers to break at paragraph boundaries, then
  sentence/line boundaries, only falling back to splitting mid-word as a last resort. This tends to
  produce more coherent chunks.
- **`CharacterTextSplitter`**: splits only on a single fixed separator (default `"\n\n"`). Simpler, but
  less adaptive — if a "paragraph" is longer than `chunk_size`, it either isn't split further or is cut
  at an arbitrary point, depending on version/settings.

> **Try it:** change `CONFIG["CHUNK_SIZE"]` to 200, 500, or 1000, and `CONFIG["CHUNK_OVERLAP"]` to 0, 50,
> or 100, then re-run the cells in this section to see how the statistics change.


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter

def compute_chunk_stats(chunks):
    '''Return a dict of summary statistics (count, avg/min/max length) for a list of Documents.'''
    lengths = [len(c.page_content) for c in chunks]
    return {
        "num_chunks": len(chunks),
        "avg_length": round(sum(lengths) / len(lengths), 1) if lengths else 0,
        "min_length": min(lengths) if lengths else 0,
        "max_length": max(lengths) if lengths else 0,
    }

def split_documents(documents, splitter):
    '''Apply a LangChain text splitter to a list of Documents and return the resulting chunks.'''
    return splitter.split_documents(documents)

def print_chunk_report(chunks, label, n_preview=3):
    '''Print summary statistics plus a preview of the first few chunks.'''
    stats = compute_chunk_stats(chunks)
    print(f"--- {label} ---")
    print(f"  Number of chunks : {stats['num_chunks']}")
    print(f"  Avg chunk length : {stats['avg_length']} chars")
    print(f"  Min chunk length : {stats['min_length']} chars")
    print(f"  Max chunk length : {stats['max_length']} chars")
    print(f"  Example chunks:")
    for i, chunk in enumerate(chunks[:n_preview]):
        preview = chunk.page_content.strip().replace(chr(10), " ")[:150]
        print(f"    [{i}] ({len(chunk.page_content)} chars) {preview}...")
    print()
    return stats

# ---- Recursive splitter (recommended default) using CONFIG values ----
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CONFIG["CHUNK_SIZE"],
    chunk_overlap=CONFIG["CHUNK_OVERLAP"],
    separators=["\n\n", "\n", ". ", " ", ""],  # tries paragraph -> line -> sentence -> word -> char
)
recursive_chunks = split_documents(all_documents, recursive_splitter)
print_chunk_report(recursive_chunks, f"RecursiveCharacterTextSplitter (size={CONFIG['CHUNK_SIZE']}, overlap={CONFIG['CHUNK_OVERLAP']})")

# ---- Character splitter for comparison ----
character_splitter = CharacterTextSplitter(
    chunk_size=CONFIG["CHUNK_SIZE"],
    chunk_overlap=CONFIG["CHUNK_OVERLAP"],
    separator="\n\n",
)
character_chunks = split_documents(all_documents, character_splitter)
print_chunk_report(character_chunks, f"CharacterTextSplitter (size={CONFIG['CHUNK_SIZE']}, overlap={CONFIG['CHUNK_OVERLAP']})")

--- RecursiveCharacterTextSplitter (size=500, overlap=50) ---
  Number of chunks : 9
  Avg chunk length : 368.0 chars
  Min chunk length : 254 chars
  Max chunk length : 492 chars
  Example chunks:
    [0] (402 chars) Chapter 4: Embeddings and Vector Search An embedding model converts a piece of text into a fixed-length numeric vector such that semantically similar ...
    [1] (259 chars) a given query vector, which is known as approximate nearest neighbor (ANN) search. Chroma is a popular open-source vector database that is easy to run...
    [2] (391 chars) Chapter 1: Introduction to Retrieval-Augmented Generation  Retrieval-Augmented Generation (RAG) is a technique that combines a retrieval system with a...

--- CharacterTextSplitter (size=500, overlap=50) ---
  Number of chunks : 8
  Avg chunk length : 414.1 chars
  Min chunk length : 254 chars
  Max chunk length : 662 chars
  Example chunks:
    [0] (662 chars) Chapter 4: Embeddings and Vector Search An embedding model converts a 

{'num_chunks': 8, 'avg_length': 414.1, 'min_length': 254, 'max_length': 662}

In [8]:
# Compare multiple chunk-size / overlap combinations side by side, without changing CONFIG.
# This helper lets students explore the effect of chunking parameters WITHOUT re-running earlier sections.

def compare_chunking_strategies(documents, size_overlap_pairs):
    '''Run RecursiveCharacterTextSplitter for each (chunk_size, chunk_overlap) pair and tabulate results.'''
    results = []
    for size, overlap in size_overlap_pairs:
        splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
        chunks = splitter.split_documents(documents)
        stats = compute_chunk_stats(chunks)
        results.append({"chunk_size": size, "chunk_overlap": overlap, **stats})
    return results

experiment_grid = [(200, 0), (200, 50), (500, 50), (500, 100), (1000, 100)]
comparison_results = compare_chunking_strategies(all_documents, experiment_grid)

print(f"{'Chunk Size':<12}{'Overlap':<10}{'# Chunks':<10}{'Avg Len':<10}{'Min Len':<10}{'Max Len'}")
print("-" * 62)
for r in comparison_results:
    print(f"{r['chunk_size']:<12}{r['chunk_overlap']:<10}{r['num_chunks']:<10}{r['avg_length']:<10}{r['min_length']:<10}{r['max_length']}")

Chunk Size  Overlap   # Chunks  Avg Len   Min Len   Max Len
--------------------------------------------------------------
200         0         29        112.8     10        198
200         50        29        120.6     10        198
500         50        9         368.0     254       492
500         100       9         368.0     254       492
1000        100       4         827.2     662       918


**Discussion questions:**

1. As `chunk_size` increases, what happens to the *number* of chunks? Why?
2. Why might a *smaller* `chunk_size` sometimes hurt retrieval quality, even though it seems more
   "precise"?
3. What is the trade-off of increasing `chunk_overlap`? (Hint: think about storage size and
   redundancy in the vector database.)
4. For our four-document demo corpus, would you expect `RecursiveCharacterTextSplitter` or
   `CharacterTextSplitter` to produce more coherent chunks? Why?

We will use `recursive_chunks` (built with the `CONFIG` values) for the rest of the notebook.


<a id="7"></a>
# 7. Embedding Models

## 7.1 What are embeddings?

An **embedding** is a fixed-length vector of real numbers that represents the *meaning* of a piece of
text. A good embedding model maps semantically similar texts to nearby points in vector space, and
dissimilar texts to distant points — even if they don't share any of the same words.

## 7.2 How semantic similarity works

Given two embedding vectors **a** and **b**, we typically measure their similarity with **cosine
similarity**:

```
cosine_similarity(a, b) = (a · b) / (||a|| * ||b||)
```

This measures the *angle* between the two vectors rather than their magnitude — two vectors pointing in
the same direction have a cosine similarity close to 1 (very similar), orthogonal vectors have a
similarity close to 0 (unrelated), and opposite vectors approach -1 (dissimilar).

## 7.3 Switchable embedding model

Just like the LLM, the embedding model is controlled by a **single config variable**:
`CONFIG["EMBEDDING_MODEL"]`. Below are four good open-source options:

| Model | Dimensions | Notes |
|---|---|---|
| `sentence-transformers/all-MiniLM-L6-v2` | 384 | Very fast, small, great default for prototyping. |
| `BAAI/bge-small-en-v1.5` | 384 | Similar size to MiniLM, generally stronger retrieval quality. |
| `BAAI/bge-base-en-v1.5` | 768 | Larger, slower, typically higher retrieval quality. |
| `intfloat/e5-small-v2` | 384 | Competitive quality; note it expects `"query: "` / `"passage: "` prefixes for best results. |

> **Tip:** Larger embedding dimensions and larger models generally give better semantic understanding,
> at the cost of more compute time and memory. For a small demo corpus like ours, `all-MiniLM-L6-v2` is
> plenty — the difference becomes more noticeable on large, nuanced corpora.


In [9]:
import time
from langchain_huggingface import HuggingFaceEmbeddings

def build_embedding_model(model_name, device):
    '''Instantiate a HuggingFaceEmbeddings wrapper around a sentence-transformers model.'''
    return HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": True},  # normalized vectors -> cosine similarity == dot product
    )

# Build the embedding model selected in CONFIG.
embedding_model = build_embedding_model(CONFIG["EMBEDDING_MODEL"], CONFIG["DEVICE"])

# Quick sanity check: embed one sentence and inspect the vector.
sample_vector = embedding_model.embed_query("What is Retrieval-Augmented Generation?")
print(f"Embedding model : {CONFIG['EMBEDDING_MODEL']}")
print(f"Vector dimension: {len(sample_vector)}")
print(f"First 8 values  : {[round(v, 4) for v in sample_vector[:8]]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model : sentence-transformers/all-MiniLM-L6-v2
Vector dimension: 384
First 8 values  : [-0.111, -0.0263, -0.0579, 0.0598, -0.0208, 0.0706, 0.0444, -0.0661]


In [10]:
# Benchmark several embedding models on our own chunk set: generation time + output dimension.
# NOTE: this downloads each model the first time it's used, so this cell can take a few minutes.

def benchmark_embedding_model(model_name, texts, device):
    '''Load an embedding model, embed a list of texts, and report timing + dimensionality.'''
    model = build_embedding_model(model_name, device)
    start = time.time()
    vectors = model.embed_documents(texts)
    elapsed = time.time() - start
    dim = len(vectors[0]) if vectors else 0
    return {
        "model": model_name,
        "num_texts": len(texts),
        "dimension": dim,
        "total_seconds": round(elapsed, 2),
        "seconds_per_text": round(elapsed / max(len(texts), 1), 4),
    }

embedding_models_to_compare = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "BAAI/bge-small-en-v1.5",
    # The two lines below are heavier -- uncomment to include them in the benchmark.
    # "BAAI/bge-base-en-v1.5",
    # "intfloat/e5-small-v2",
]

sample_texts = [c.page_content for c in recursive_chunks[:10]]  # use up to 10 chunks for a quick benchmark

embedding_benchmark_results = []
for model_name in embedding_models_to_compare:
    print(f"Benchmarking {model_name} ...")
    result = benchmark_embedding_model(model_name, sample_texts, CONFIG["DEVICE"])
    embedding_benchmark_results.append(result)

print(f"\n{'Model':<45}{'Dim':<8}{'Total (s)':<12}{'Sec/text'}")
print("-" * 80)
for r in embedding_benchmark_results:
    print(f"{r['model']:<45}{r['dimension']:<8}{r['total_seconds']:<12}{r['seconds_per_text']}")

Benchmarking sentence-transformers/all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Benchmarking BAAI/bge-small-en-v1.5 ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Model                                        Dim     Total (s)   Sec/text
--------------------------------------------------------------------------------
sentence-transformers/all-MiniLM-L6-v2       384     0.15        0.0171
BAAI/bge-small-en-v1.5                       384     0.09        0.0095


**Approximate memory footprint (parameter count, for reference — not measured live to keep this
cell fast):**

| Model | Approx. parameters | Approx. disk size |
|---|---|---|
| `all-MiniLM-L6-v2` | ~22M | ~90 MB |
| `bge-small-en-v1.5` | ~33M | ~130 MB |
| `bge-base-en-v1.5` | ~109M | ~440 MB |
| `e5-small-v2` | ~33M | ~130 MB |

**Expected retrieval quality (general guidance, not a guarantee for every corpus):** larger, more
recent models (`bge-base-en-v1.5`) tend to outperform smaller ones (`all-MiniLM-L6-v2`) on standard
retrieval benchmarks (e.g., MTEB), but for small, narrow-domain corpora like our demo dataset, the gap
is often barely noticeable. Always validate on *your own* data and queries.


<a id="8"></a>
# 8. Vector Database

## 8.1 What a vector database does

A vector database stores embedding vectors (plus their original text and metadata) and provides fast
**similarity search**: given a query vector, it returns the *k* stored vectors closest to it. We use
**Chroma**, a lightweight, open-source vector database that runs entirely locally (no server, no API
key) and persists to disk.

## 8.2 Cosine similarity, recap

Because we normalized our embeddings in Section 7 (`normalize_embeddings=True`), the **dot product**
between two vectors is mathematically equivalent to their **cosine similarity**. Chroma's default
distance metric is squared L2 distance, but for normalized vectors, ranking by L2 distance and ranking
by cosine similarity produce the *same order* — smaller distance = higher similarity.

## 8.3 What we'll do

1. Create a fresh Chroma collection from our `recursive_chunks`.
2. Persist it to disk (`CONFIG["PERSIST_DIRECTORY"]`).
3. Reload it from disk, to prove persistence works.
4. Run a similarity search and inspect the retrieved chunks, their similarity scores, and metadata.


In [11]:
from langchain_chroma import Chroma
import shutil

def build_vector_store(chunks, embedding_model, persist_directory, collection_name="rag_demo"):
    '''Create a new Chroma vector store from a list of chunks and persist it to disk.'''
    # Start from a clean directory each time this function is called, so re-running the notebook
    # doesn't silently accumulate duplicate chunks across runs.
    if os.path.exists(persist_directory):
        shutil.rmtree(persist_directory)
    os.makedirs(persist_directory, exist_ok=True)

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory=persist_directory,
        collection_name=collection_name,
    )
    return vector_store

def load_vector_store(embedding_model, persist_directory, collection_name="rag_demo"):
    '''Reload an already-persisted Chroma vector store from disk.'''
    return Chroma(
        embedding_function=embedding_model,
        persist_directory=persist_directory,
        collection_name=collection_name,
    )

# 1. Build and persist.
vector_store = build_vector_store(recursive_chunks, embedding_model, CONFIG["PERSIST_DIRECTORY"])
print(f"Vector store created with {vector_store._collection.count()} vectors, "
      f"persisted at '{CONFIG['PERSIST_DIRECTORY']}'.")

# 2. Reload from disk, to demonstrate persistence.
reloaded_vector_store = load_vector_store(embedding_model, CONFIG["PERSIST_DIRECTORY"])
print(f"Reloaded vector store contains {reloaded_vector_store._collection.count()} vectors.")

# From here on, use the reloaded store (proves the pipeline works even after a fresh restart).
vector_store = reloaded_vector_store

Vector store created with 9 vectors, persisted at './chroma_db'.
Reloaded vector store contains 9 vectors.


In [12]:
def similarity_search_with_scores(vector_store, query, k=4):
    '''Run a similarity search and return (Document, score) pairs, plus print a readable report.'''
    results = vector_store.similarity_search_with_relevance_scores(query, k=k)
    print(f"Query: '{query}'  (top {k} results)\n")
    for rank, (doc, score) in enumerate(results, start=1):
        preview = doc.page_content.strip().replace(chr(10), " ")[:150]
        print(f"  #{rank}  score={score:.4f}  source={doc.metadata.get('source', 'unknown')}")
        print(f"       {preview}...\n")
    return results

_ = similarity_search_with_scores(vector_store, "What is Retrieval-Augmented Generation?", k=CONFIG["TOP_K"])

Query: 'What is Retrieval-Augmented Generation?'  (top 4 results)

  #1  score=0.5413  source=data/01_introduction.txt
       Chapter 1: Introduction to Retrieval-Augmented Generation  Retrieval-Augmented Generation (RAG) is a technique that combines a retrieval system with a...

  #2  score=0.3000  source=data/01_introduction.txt
       RAG was popularized by the 2020 paper "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks" by Lewis et al., which showed that combining ...

  #3  score=0.1439  source=data/02_advantages.md
       Chapter 2: Advantages of RAG  RAG offers several concrete advantages over relying on a language model's parametric memory alone:  Up-to-date knowledge...

  #4  score=0.0546  source=data/02_advantages.md
       Lower cost than fine-tuning: updating a vector database is far cheaper than retraining or fine-tuning a large model.  Limitations  RAG is not a silver...



**Note on scores:** LangChain's `similarity_search_with_relevance_scores` returns a score in
roughly the `[0, 1]` range (higher = more similar) after normalizing Chroma's raw distance. If you ever
see scores outside that range, it usually means the relevance-score normalization function assumed a
different distance metric than the one actually configured — a good example of why understanding what a
library does *under the hood* matters, not just calling it.


<a id="9"></a>
# 9. Retriever

## 9.1 From vector store to retriever

A **retriever** is a thin, standardized wrapper around a search mechanism (here, our Chroma vector
store) that exposes a single method, `invoke(query)`, returning a list of `Document`s. Wrapping the
vector store as a retriever lets it be plugged directly into a LangChain pipeline (Section 12) using the
same interface as other retrieval methods (e.g., BM25, MMR — see Section 17 bonus challenges).

## 9.2 Precision vs. recall in retrieval

- **`top_k` too small** → high **precision** (few irrelevant chunks) but low **recall** (the correct
  chunk might get missed entirely).
- **`top_k` too large** → high **recall** (correct chunk very likely included) but low **precision**
  (LLM has to sift through more irrelevant context, which can dilute the answer or exceed the context
  window).

There is no universally "correct" `top_k` — it depends on chunk size, corpus size, and how spread out
the relevant information is.


In [13]:
def build_retriever(vector_store, k):
    '''Wrap a vector store as a LangChain retriever configured for similarity search with top-k results.'''
    return vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k},
    )

def show_retrieved_contexts(retriever, query):
    '''Invoke a retriever and pretty-print the retrieved chunks.'''
    docs = retriever.invoke(query)
    print(f"Query: '{query}'  ->  {len(docs)} chunk(s) retrieved\n")
    for i, doc in enumerate(docs, start=1):
        preview = doc.page_content.strip().replace(chr(10), " ")[:150]
        print(f"  [{i}] source={doc.metadata.get('source', 'unknown')}")
        print(f"      {preview}...\n")
    return docs

retriever = build_retriever(vector_store, CONFIG["TOP_K"])
_ = show_retrieved_contexts(retriever, "What are the advantages of RAG?")

Query: 'What are the advantages of RAG?'  ->  4 chunk(s) retrieved

  [1] source=data/02_advantages.md
      Chapter 2: Advantages of RAG  RAG offers several concrete advantages over relying on a language model's parametric memory alone:  Up-to-date knowledge...

  [2] source=data/01_introduction.txt
      The main components of a RAG system are: a document loader, a text splitter, an embedding model, a vector database, a retriever, a prompt template, an...

  [3] source=data/01_introduction.txt
      Chapter 1: Introduction to Retrieval-Augmented Generation  Retrieval-Augmented Generation (RAG) is a technique that combines a retrieval system with a...

  [4] source=data/01_introduction.txt
      RAG was popularized by the 2020 paper "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks" by Lewis et al., which showed that combining ...



In [14]:
# Compare different top-k values on the SAME query, without touching CONFIG.

def compare_top_k(vector_store, query, k_values):
    '''Run the same query at several top-k values and report how many unique sources are covered.'''
    print(f"Query: '{query}'\n")
    for k in k_values:
        temp_retriever = build_retriever(vector_store, k)
        docs = temp_retriever.invoke(query)
        unique_sources = {d.metadata.get("source", "unknown") for d in docs}
        print(f"  k={k:<3} -> {len(docs)} chunks retrieved from {len(unique_sources)} unique source file(s)")

compare_top_k(vector_store, "What are the advantages of RAG?", k_values=[2, 4, 6, 8])

Query: 'What are the advantages of RAG?'

  k=2   -> 2 chunks retrieved from 2 unique source file(s)
  k=4   -> 4 chunks retrieved from 2 unique source file(s)
  k=6   -> 6 chunks retrieved from 3 unique source file(s)
  k=8   -> 8 chunks retrieved from 4 unique source file(s)


**Discussion:** As `k` grows, do you see the retriever start pulling in chunks from *irrelevant*
source files? At what point does that start happening for this corpus? This is precision degrading as
recall increases — exactly the trade-off described above.


<a id="10"></a>
# 10. Prompt Engineering

## 10.1 Why prompt wording matters

The prompt is the *only* channel through which the LLM sees the retrieved context and the question. Its
wording strongly influences whether the model:

- Sticks strictly to the provided context (vs. falling back on parametric memory / hallucinating).
- Says "I don't know" when the context is insufficient (vs. guessing).
- Formats the answer usefully (concise vs. rambling, with/without citations).

## 10.2 A clean RAG prompt template

We use LangChain's `PromptTemplate` to define a reusable template with two placeholders: `{context}`
(the retrieved chunks, concatenated) and `{question}` (the user's question). Explicit instructions tell
the model to only use the given context and to admit when it doesn't know the answer.

> **Try it:** modify `RAG_PROMPT_TEMPLATE` below — e.g., ask for a one-sentence answer, or ask the model
> to cite which source file it used — and re-run Section 12 to see the effect.


In [15]:
from langchain_core.prompts import PromptTemplate

RAG_PROMPT_TEMPLATE = '''You are a helpful teaching assistant. Answer the question using ONLY the
context provided below. If the context does not contain enough information to answer, say
"I don't have enough information in the provided context to answer that." Do not make up information
that is not in the context.

Context:
{context}

Question: {question}

Answer:'''

rag_prompt = PromptTemplate(
    template=RAG_PROMPT_TEMPLATE,
    input_variables=["context", "question"],
)

def format_docs_for_prompt(docs):
    '''Concatenate retrieved Documents into a single context string, one chunk per block, with source tags.'''
    blocks = []
    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source", "unknown")
        blocks.append(f"[Chunk {i} | source: {source}]\n{doc.page_content}")
    return "\n\n".join(blocks)

# Preview: build and print a fully-formatted prompt WITHOUT calling the LLM yet.
example_question = "What are the advantages of RAG?"
example_docs = retriever.invoke(example_question)
example_context = format_docs_for_prompt(example_docs)
example_prompt_text = rag_prompt.format(context=example_context, question=example_question)

print("=" * 80)
print("EXAMPLE PROMPT THAT WILL BE SENT TO THE LLM")
print("=" * 80)
print(example_prompt_text)

EXAMPLE PROMPT THAT WILL BE SENT TO THE LLM
You are a helpful teaching assistant. Answer the question using ONLY the
context provided below. If the context does not contain enough information to answer, say
"I don't have enough information in the provided context to answer that." Do not make up information
that is not in the context.

Context:
[Chunk 1 | source: data/02_advantages.md]
Chapter 2: Advantages of RAG

RAG offers several concrete advantages over relying on a language model's parametric memory alone:

Up-to-date knowledge: the knowledge base can be updated at any time without retraining the model.

Source attribution: because retrieved chunks carry metadata (source file, page number), answers can be traced back to their origin.

Reduced hallucination: grounding generation in retrieved text reduces (but does not eliminate) fabricated answers.

[Chunk 2 | source: data/01_introduction.txt]
The main components of a RAG system are: a document loader, a text splitter, an embedding

<a id="11"></a>
# 11. LLM

## 11.1 Loading the instruction-tuned model

We load the model named in `CONFIG["LLM_MODEL"]` with HuggingFace `transformers`, wrap it in a
`transformers.pipeline`, and expose it to LangChain via `HuggingFacePipeline`. Changing the model used
by the *entire* notebook only requires editing `CONFIG["LLM_MODEL"]` in Section 3.

| Model | Approx. size | Speed | Quality | Notes |
|---|---|---|---|---|
| `TinyLlama/TinyLlama-1.1B-Chat-v1.0` | 1.1B params | Fastest, runs fine on CPU | Basic but coherent | **Recommended default** for this assignment — fits comfortably in Colab's free tier. |
| `Qwen/Qwen2.5-1.5B-Instruct` | 1.5B params | Fast | Noticeably better instruction following | Good middle ground. |
| `google/gemma-2-2b-it` | 2B params | Slower, benefits strongly from GPU | Best quality of the three | Requires accepting the model's license on HuggingFace and, for gated access, a HF token. |

## 11.2 Key generation parameters

| Parameter | Meaning | Effect |
|---|---|---|
| `temperature` | Randomness of sampling. | Low (e.g. 0.1–0.3) → focused, deterministic-ish answers. High (e.g. 0.8–1.2) → more creative/varied, but riskier for factual QA. |
| `max_new_tokens` | Maximum number of tokens to generate. | Higher → longer answers possible, but slower and more likely to ramble. |
| `do_sample` | Whether to sample at all. | `False` → greedy decoding (always picks the most likely token; fully deterministic). `True` → enables `temperature`/`top_p` sampling. |
| `top_p` | Nucleus sampling threshold. | Keeps only the smallest set of tokens whose cumulative probability exceeds `top_p`; lower values → more focused output. |
| `repetition_penalty` | Penalizes tokens that already appeared. | >1.0 discourages the model from repeating phrases/looping. |

For factual RAG question-answering, we recommend **low temperature** (0.1–0.4) and **`do_sample=True`
with a moderate `top_p`** — enough to avoid robotic repetition, but focused enough to stay grounded in
the retrieved context.


In [16]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline

def load_llm(model_name, device, temperature, max_new_tokens, do_sample, top_p, repetition_penalty):
    '''Load an instruction-tuned HuggingFace causal LM and wrap it as a LangChain-compatible LLM.'''
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype="auto",
        device_map="auto" if device == "cuda" else None,
    )
    if device == "cpu":
        model = model.to("cpu")

    text_gen_pipeline = pipeline(
        task="text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=do_sample,
        top_p=top_p,
        repetition_penalty=repetition_penalty,
        return_full_text=False,   # only return the newly generated text, not the prompt echoed back
    )
    return HuggingFacePipeline(pipeline=text_gen_pipeline)

# Load the LLM selected in CONFIG. This downloads the model on first run (may take a few minutes).
llm = load_llm(
    model_name=CONFIG["LLM_MODEL"],
    device=CONFIG["DEVICE"],
    temperature=CONFIG["TEMPERATURE"],
    max_new_tokens=CONFIG["MAX_NEW_TOKENS"],
    do_sample=CONFIG["DO_SAMPLE"],
    top_p=CONFIG["TOP_P"],
    repetition_penalty=CONFIG["REPETITION_PENALTY"],
)

print(f"Loaded LLM: {CONFIG['LLM_MODEL']} on device={CONFIG['DEVICE']}")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample', 'top_p', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Loaded LLM: TinyLlama/TinyLlama-1.1B-Chat-v1.0 on device=cuda


In [17]:
# Quick sanity check: generate a short answer with NO retrieval context at all,
# just to confirm the model loads and produces text.
quick_test_response = llm.invoke("In one sentence, what is machine learning?")
print("LLM raw output:\n")
print(quick_test_response)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


LLM raw output:




**Common mistake:** if you see `CUDA out of memory`, either switch to a smaller model
(`TinyLlama`), reduce `MAX_NEW_TOKENS`, or enable 4-bit quantization via `bitsandbytes`
(`load_in_4bit=True` in `AutoModelForCausalLM.from_pretrained`, which we installed in Section 2 but did
not enable by default to keep the code simple).


<a id="12"></a>
# 12. Building the Complete RAG Pipeline

## 12.1 Putting it all together

We now assemble every component built so far — retriever, prompt template, and LLM — into one runnable
pipeline using **LangChain Expression Language (LCEL)**, the modern, non-deprecated way to compose
LangChain components with the `|` (pipe) operator.

```
   question
      │
      ▼
 [ retriever ]  ──►  retrieved Documents ──► format_docs_for_prompt() ──► context string
      │                                                                        │
      └────────────────────────────► {question}                               │
                                          │                                    │
                                          ▼                                    ▼
                                     rag_prompt.format(context, question) ──► full prompt
                                                          │
                                                          ▼
                                                        [ llm ]
                                                          │
                                                          ▼
                                                        answer
```

We wrap this into a single reusable function, `ask_rag()`, so every later section can call one function
instead of repeating the wiring.


In [18]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def build_rag_chain(retriever, prompt, llm):
    '''Compose retriever + prompt + LLM into a single LCEL runnable chain.'''
    rag_chain = (
        RunnableParallel(
            {
                "context": retriever | format_docs_for_prompt,   # retrieve docs, then join into a string
                "question": RunnablePassthrough(),                # pass the question through unchanged
            }
        )
        | prompt
        | llm
        | StrOutputParser()
    )
    return rag_chain

def ask_rag(question, retriever, prompt, llm, verbose=True):
    '''Run the full RAG pipeline end-to-end for a single question and return the answer plus sources.'''
    docs = retriever.invoke(question)
    context = format_docs_for_prompt(docs)
    full_prompt = prompt.format(context=context, question=question)
    raw_answer = llm.invoke(full_prompt)
    answer = raw_answer.strip()

    if verbose:
        print(f"Q: {question}\n")
        print(f"A: {answer}\n")
        print("Sources used:")
        for doc in docs:
            print(f"  - {doc.metadata.get('source', 'unknown')}")
        print()

    return {"question": question, "answer": answer, "source_documents": docs}

# Build the end-to-end chain (also usable directly via rag_chain.invoke(question)).
rag_chain = build_rag_chain(retriever, rag_prompt, llm)

# Run one full end-to-end example through our convenience function.
result = ask_rag("What is Retrieval-Augmented Generation?", retriever, rag_prompt, llm)

Q: What is Retrieval-Augmented Generation?

A: Retrieval-Augmented Generation (RAG) is a technique that combines a retrieval system with a generative language model for improving the quality of generated responses.

Sources used:
  - data/01_introduction.txt
  - data/01_introduction.txt
  - data/02_advantages.md
  - data/02_advantages.md



**Common mistake:** if the answer looks like it ignores the retrieved context entirely, double
check that `format_docs_for_prompt` is actually being called on the retriever's output (not the raw
`Document` objects) before it reaches the prompt template — LLMs will often just describe a Python
object repr if handed one by accident.


<a id="13"></a>
# 13. Experiments

## 13.1 Predefined questions

Run the full pipeline over a small, fixed set of questions. Some are directly answerable from the
corpus; others (e.g., "Who is the author?") test whether the model correctly uses **only** the provided
context instead of guessing.


In [19]:
EXPERIMENT_QUESTIONS = [
    "What is Retrieval-Augmented Generation?",
    "Summarize Chapter 2.",
    "List all advantages mentioned.",
    "What chunking methods were used?",
    "Who is the author?",
]

def run_experiment_batch(questions, retriever, prompt, llm):
    '''Run ask_rag() over a list of questions and collect the results into a list of dicts.'''
    batch_results = []
    for q in questions:
        result = ask_rag(q, retriever, prompt, llm, verbose=False)
        batch_results.append(result)
        print(f"Q: {result['question']}")
        print(f"A: {result['answer']}")
        print("-" * 80)
    return batch_results

baseline_results = run_experiment_batch(EXPERIMENT_QUESTIONS, retriever, rag_prompt, llm)

Q: What is Retrieval-Augmented Generation?
A: Retrieval-augmented generation (RAG) is a technique that combines a retrieval system with a generative language model for generating answers to questions.
--------------------------------------------------------------------------------
Q: Summarize Chapter 2.
A: The chapter discusses how RAG offers up-to-date knowledge, source attribution, reduced hallucination, and a combination of retrieval and generative methods.
--------------------------------------------------------------------------------
Q: List all advantages mentioned.
A: Chapter 2: Advantages of RAG
- Up-to-date knowledge
- Source attribution
- Reduced hallucination
- Modularity
--------------------------------------------------------------------------------
Q: What chunking methods were used?
A: Chunking strategies, such as RAW and RAG, split long documents into smaller pieces before embedding them. The size and overlap of these chunks control the precision and recall of the ret

## 13.2 Controlled comparisons

The cell below lets you sweep ONE variable at a time (chunk size, chunk overlap, top-k, embedding model,
prompt, or LLM) while holding everything else fixed, and compare the resulting answer for a single fixed
question. This is the core experimental tool for Section 15's experiment table.

> **Note:** sweeping `EMBEDDING_MODEL` or `LLM_MODEL` reloads a model each time, which can be slow.
> Sweeping `CHUNK_SIZE`, `CHUNK_OVERLAP`, or `TOP_K` is much faster since it reuses the already-loaded
> models.


In [20]:
def run_chunk_size_sweep(question, chunk_sizes, chunk_overlap, documents, embedding_model, llm, prompt, k):
    '''Rebuild the vector store for each chunk_size and compare the resulting answer for one question.'''
    sweep_results = []
    for size in chunk_sizes:
        splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=chunk_overlap)
        chunks = splitter.split_documents(documents)
        temp_store = build_vector_store(chunks, embedding_model, persist_directory=f"./tmp_chroma_{size}")
        temp_retriever = build_retriever(temp_store, k)
        result = ask_rag(question, temp_retriever, prompt, llm, verbose=False)
        sweep_results.append({"chunk_size": size, "num_chunks": len(chunks), "answer": result["answer"]})
    return sweep_results

sweep_question = "What are the advantages of RAG?"
chunk_size_sweep = run_chunk_size_sweep(
    question=sweep_question,
    chunk_sizes=[200, 500, 1000],
    chunk_overlap=CONFIG["CHUNK_OVERLAP"],
    documents=all_documents,
    embedding_model=embedding_model,
    llm=llm,
    prompt=rag_prompt,
    k=CONFIG["TOP_K"],
)

print(f"Question: '{sweep_question}'\n")
for r in chunk_size_sweep:
    print(f"chunk_size={r['chunk_size']} ({r['num_chunks']} chunks)")
    print(f"  Answer: {r['answer']}\n")

Question: 'What are the advantages of RAG?'

chunk_size=200 (29 chunks)
  Answer: RAG offers several concrete advantages over relying on a language model's parametric memory alone, including a modular design, the ability to swap components independently, and the potential for improving retrieval performance by incorporating additional context.

chunk_size=500 (9 chunks)
  Answer: RAG offers several concrete advantages over relying on a language model's parametric memory alone:

1. Up-to-date knowledge: the knowledge base can be updated at any time without retraining the model.
2. Source attribution: because retrieved chunks carry metadata (source file, page number), answers can be traced back to their origin.
3. Reduced hallucination: grounding generation in retrieved text reduces (but does not eliminate) fabricated answers.

chunk_size=1000 (4 chunks)
  Answer: RAG offers several concrete advantages over relying on a language model's parametric memory alone:

- Up-to-date knowledge: t

**Reflect:** Did the answer content noticeably change across chunk sizes for this question? If
not, is that because the question is "easy" (the relevant fact appears clearly regardless of chunking),
or because our demo corpus is too small to show a strong effect? Try the same sweep with a *harder*
question — e.g. "What chunking methods were used?" — and compare.


<a id="14"></a>
# 14. Optional Evaluation

> **This entire section is OPTIONAL.** You are not required to run any automated evaluation framework
> to complete this assignment — the manual evaluation table (Option C) is sufficient and is what
> Section 15 expects you to fill in.

## 14.1 Why evaluating RAG is hard

Unlike a classification model, a RAG system doesn't have one "correct" output to compare against with
simple accuracy. There are actually **two things** to evaluate, which can each go wrong independently:

1. **Retrieval quality** — did we fetch the *right* chunks?
2. **Generation quality** — given those chunks, did the LLM produce a *faithful*, *relevant*, well
   written answer?

A great retriever with a poor generator (or vice versa) can still produce a bad final answer, so
end-to-end accuracy alone doesn't tell you *which* component to fix.

## 14.2 Option A — RAGAS (reference-free RAG metrics)

[RAGAS](https://github.com/explodinggradients/ragas) computes metrics like:

- **Faithfulness** — is the generated answer actually supported by the retrieved context (or is it
  hallucinating beyond it)?
- **Answer Relevancy** — does the answer actually address the question asked?
- **Context Precision** — of the retrieved chunks, how many were actually relevant?
- **Context Recall** — of the relevant information in the corpus, how much did we retrieve?

The cell below is wrapped in a `try/except` because RAGAS typically expects an LLM-as-judge (often an
OpenAI model) to compute some of its metrics, which conflicts with this notebook's "open-source only,
no paid API" requirement. We show the code for completeness and let it fail gracefully if RAGAS or its
dependencies aren't configured for a fully local judge model.

## 14.3 Option B — DeepEval

[DeepEval](https://github.com/confident-ai/deepeval) is a similar alternative RAG-evaluation library
with metrics like `FaithfulnessMetric` and `ContextualRelevancyMetric`. Like RAGAS, it typically expects
an LLM judge; you would configure it to use our local `llm` object instead of a paid API. We do not run
it by default in this notebook to keep runtime short, but the pattern is identical to Option A.

## 14.4 Option C — Manual evaluation table (recommended for this assignment)

The most reliable evaluation method at this scale is simply reading the retrieved context and answer
yourself, and judging correctness. This is exactly what Section 15's experiment table is for.


In [21]:
# OPTIONAL: attempt an automated RAGAS evaluation. This is wrapped in try/except because RAGAS
# generally expects an LLM judge (often configured for a paid API) to compute several metrics, which
# this open-source-only notebook does not set up by default.
try:
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy
    from datasets import Dataset

    eval_question = "What is Retrieval-Augmented Generation?"
    eval_result = ask_rag(eval_question, retriever, rag_prompt, llm, verbose=False)

    ragas_dataset = Dataset.from_dict({
        "question": [eval_result["question"]],
        "answer": [eval_result["answer"]],
        "contexts": [[d.page_content for d in eval_result["source_documents"]]],
    })

    # NOTE: evaluate() below will likely need an LLM/embeddings judge explicitly configured to run
    # fully offline. This is left as an extension exercise -- see the RAGAS docs for local LLM judges.
    scores = evaluate(ragas_dataset, metrics=[faithfulness, answer_relevancy])
    print(scores)
except Exception as e:
    print("RAGAS evaluation skipped (this is expected in a fully local/offline setup).")
    print(f"Reason: {e}")
    print("\nThis is OPTIONAL -- proceed to the manual evaluation table below instead.")

RAGAS evaluation skipped (this is expected in a fully local/offline setup).
Reason: No module named 'ragas'

This is OPTIONAL -- proceed to the manual evaluation table below instead.


In [22]:
# Manual evaluation table (Option C) -- fill this in by hand after reviewing answers.
import pandas as pd

manual_eval_rows = [
    {
        "Question": "What is Retrieval-Augmented Generation?",
        "Expected Answer": "A technique combining retrieval with generation to ground LLM answers in external documents.",
        "Retrieved Context": "(fill in after inspecting source_documents)",
        "Generated Answer": "(fill in from ask_rag output)",
        "Correct?": "",
        "Comments": "",
    },
    {
        "Question": "Who is the author?",
        "Expected Answer": "NLP Teaching Team (stated in the PDF).",
        "Retrieved Context": "",
        "Generated Answer": "",
        "Correct?": "",
        "Comments": "",
    },
]

manual_eval_df = pd.DataFrame(manual_eval_rows)
manual_eval_df

,Question,Expected Answer,Retrieved Context,Generated Answer,Correct?,Comments
0,What is Retrieval-Augmented Generation?,A technique combining retrieval with generatio...,(fill in after inspecting source_documents),(fill in from ask_rag output),,
1,Who is the author?,NLP Teaching Team (stated in the PDF).,,,,


<a id="15"></a>
# 15. Student Experiment Table

Run at least **5 different configurations** by changing `CONFIG` values (or using the sweep helpers from
Section 13) and record your observations below. Duplicate this table as needed.

| Experiment | Chunking | Chunk Size | Overlap | Embedding | LLM | k | Observation |
|---|---|---|---|---|---|---|---|
| 1 | Recursive | 500 | 50 | all-MiniLM-L6-v2 | TinyLlama-1.1B-Chat | 4 | *(baseline — fill in)* |
| 2 | Recursive | 200 | 0  | all-MiniLM-L6-v2 | TinyLlama-1.1B-Chat | 4 | |
| 3 | Recursive | 1000| 100| all-MiniLM-L6-v2 | TinyLlama-1.1B-Chat | 4 | |
| 4 | Recursive | 500 | 50 | bge-small-en-v1.5 | TinyLlama-1.1B-Chat | 4 | |
| 5 | Recursive | 500 | 50 | all-MiniLM-L6-v2 | TinyLlama-1.1B-Chat | 8 | |
| 6 | Character | 500 | 50 | all-MiniLM-L6-v2 | TinyLlama-1.1B-Chat | 4 | |
| 7 | Recursive | 500 | 50 | all-MiniLM-L6-v2 | Qwen2.5-1.5B-Instruct | 4 | |

> **Tip:** Use `run_experiment_batch()` (Section 13) or `run_chunk_size_sweep()` after adjusting
> `CONFIG` to quickly regenerate answers for each row.


<a id="16"></a>
# 16. Reflection Questions

Answer the following in your own words (2–4 sentences each), based on the experiments you ran above.

1. Which embedding model performed best on your queries, and how did you judge "best"?
2. How did increasing chunk size affect the relevance of retrieved context? Did larger chunks help or
   hurt?
3. How did chunk overlap influence the coherence of the final answers?
4. Which chunking strategy (`RecursiveCharacterTextSplitter` vs `CharacterTextSplitter`) produced more
   coherent chunks for this corpus, and why do you think that is?
5. Which retriever `top_k` setting gave the best balance of precision and recall for your test
   questions?
6. Describe a case where retrieval **failed** — i.e., the retriever returned chunks that were not
   actually relevant to the question. What made that query hard?
7. If you were to improve this pipeline for a *real* production use case, what would you change first,
   and why?
8. Of the four building blocks — chunking, embeddings, retrieval, and the LLM itself — which one had the
   largest observed impact on final answer quality in your experiments? Justify your answer with a
   specific example.


<a id="17"></a>
# 17. Bonus Challenges

These are optional, ungraded (unless your instructor says otherwise) extensions. Each includes a short
starting hint — you are expected to fill in the implementation yourself.

## 17.1 Add an HTML loader

```python
from langchain_community.document_loaders import BSHTMLLoader
html_docs = BSHTMLLoader("path/to/file.html").load()
```

## 17.2 Add a CSV loader

```python
from langchain_community.document_loaders import CSVLoader
csv_docs = CSVLoader("path/to/file.csv").load()
```

## 17.3 Add a JSON loader

```python
from langchain_community.document_loaders import JSONLoader
json_docs = JSONLoader("path/to/file.json", jq_schema=".[]", text_content=False).load()
```

## 17.4 Replace Chroma with FAISS

```python
from langchain_community.vectorstores import FAISS
faiss_store = FAISS.from_documents(recursive_chunks, embedding_model)
faiss_store.save_local("./faiss_index")
```
Compare query latency and result overlap against the Chroma-based `vector_store` from Section 8.

## 17.5 Implement BM25 retrieval

```python
from langchain_community.retrievers import BM25Retriever
bm25_retriever = BM25Retriever.from_documents(recursive_chunks)
bm25_retriever.k = CONFIG["TOP_K"]
```
BM25 is a classic *keyword*-based retriever (no embeddings involved) — compare its results to the
embedding-based retriever for queries with exact keyword overlap vs. queries that require semantic
understanding.

## 17.6 Implement MMR (Maximal Marginal Relevance) retrieval

```python
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": CONFIG["TOP_K"], "fetch_k": 20, "lambda_mult": 0.5},
)
```
MMR balances relevance against *diversity*, reducing redundant near-duplicate chunks in the retrieved
set.

## 17.7 Compare Similarity vs. MMR

Run the same question through both `retriever` (similarity) and `mmr_retriever` (MMR) and compare the
diversity of the retrieved chunks' `source` metadata.

## 17.8 Add source citations to answers

Modify `RAG_PROMPT_TEMPLATE` (Section 10) to instruct the model to cite `[Chunk N]` markers inline, and
post-process the answer to map those back to `doc.metadata["source"]`.

## 17.9 Display retrieved documents in a nicer format

Use `pandas.DataFrame` to render retrieved chunks (source, score, preview) as a table instead of printed
text.

## 17.10 Build a simple Gradio interface

```python
%pip install -q gradio
import gradio as gr

def gradio_ask(question):
    result = ask_rag(question, retriever, rag_prompt, llm, verbose=False)
    return result["answer"]

gr.Interface(fn=gradio_ask, inputs="text", outputs="text", title="RAG Demo").launch(share=True)
```

## 17.11 Build a simple Streamlit interface

Streamlit apps run as standalone scripts rather than in-notebook, so this challenge involves exporting
your pipeline to a `app.py` file with `st.text_input` for the question and `st.write` for the answer,
then running `streamlit run app.py` (locally, not inside Colab).

---

**End of assignment.** Make sure you have filled in Section 15 (Experiment Table) and Section 16
(Reflection Questions) before submitting.
